# Now we will enter the training phase

## 1. Dataloading

But before we can train of our preprocessed dataset, we need to load it into a trainable format ( Pytorch Dataset )

And to do that we will create a seperated script that creates a class inhereting (subclassing ) from the Torch class Dataset `torch.utils.data.Dataset class` which primarily involves implementing three key methods:

- `__init__` : initialization, its excecuted at the creation of the dataset object ( for our case we will a path variable )
- `__len__` : Total Number of Samples, used particulary for batching
- `__getitem__` : Retrieve a single sample based on the index provided (`idx`)

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [4]:
import torch
from torch.utils.data import DataLoader
from scripts.dataset import PortfolioDataset,multitask_collate_fn

train_dataset = PortfolioDataset(
    "../data/cleaned/processed_mdeberta_v3_base/train_processed.jsonl"
    ) # this uses the __init__ method of PortfolioDataset to load the dataset from the specified JSONL file. It should read the file and prepare the data for training.
validation_dataset = PortfolioDataset(
    "../data/cleaned/processed_mdeberta_v3_base/val_processed.jsonl"
    )
train_dataset[0] # This uses the __getitem__ method of PortfolioDataset to get the first item in the dataset. It should return a dictionary with keys like "input_ids", "attention_mask", "labels", etc., depending on how the dataset is structured.

{'input_ids': tensor([     1,    321,  21071,  91916,   6163,    270,  32329,    625,    260,
          81291,    764,   2654,  69376,    866,    260,    267,  18515,    450,
           8987,    260,   1531,    442,  21687,    260,  41469,    362,    262,
            327,  34759,   6612,  16243,    562,    329,  34072,    290,    260,
            266,  34244,  86804, 129774,    262,    451,    380,  34597,    318,
         218679,    362,    625,  14186,    266,  63181,    321,    488, 185471,
           2443,  72157,    901,    262,    337,  23738,   7781,   6874,    866,
            556, 195335,    586,  94827,    261,      2]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 'ner_labels': tensor([-100,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            

We will now turn our Dataset into a stream of batches that can be interable for more training efficienct (the GPU will be able to process many samples in parallel),
And also it helps shuffling our data, And with `num_workers` we allow our data to be loaded to the GPU while its training we can further optimize it allowing `pin_memory` to reduce cpu_gpu batch transfer latency.

In [5]:
train_dataloader = DataLoader(train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=multitask_collate_fn,
    num_workers=2,
    pin_memory=True,)

validation_dataloader = DataLoader(validation_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=multitask_collate_fn,
    num_workers=2,
    pin_memory=True,)

batch = next(iter(train_dataloader))
for k, v in batch.items():
    if hasattr(v, "shape"):
        print(k, v.shape)

input_ids torch.Size([8, 94])
attention_mask torch.Size([8, 94])
ner_labels torch.Size([8, 94])
domain_labels torch.Size([8, 10])


## 2. Model loading and environnement

In [6]:
# Setup device agnostic code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


---------- TEST POUR PRODUCTION ------------

In [7]:
from transformers import AutoTokenizer, AutoModel

encoder_name = "microsoft/mdeberta-v3-base"  # or whichever you pick

# Both must come from the same model name
tokenizer = AutoTokenizer.from_pretrained(encoder_name)
model = AutoModel.from_pretrained(encoder_name)

c:\Users\LCELIL\projet-integration\ai\venv\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\LCELIL\projet-integration\ai\venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\LCELIL\projet-integration\ai\venv\Lib\site-packages\transformers\convert_slow_tokenizer.py:473: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unkno

pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

----------------------

In [8]:
# Example to print the names of each layer block
for name, layer in model.named_children():
    print(name)

embeddings
encoder


In [9]:
# Check which parameters are trainable
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Trainable layer: {name}")

Trainable layer: embeddings.word_embeddings.weight
Trainable layer: embeddings.LayerNorm.weight
Trainable layer: embeddings.LayerNorm.bias
Trainable layer: encoder.layer.0.attention.self.query_proj.weight
Trainable layer: encoder.layer.0.attention.self.query_proj.bias
Trainable layer: encoder.layer.0.attention.self.key_proj.weight
Trainable layer: encoder.layer.0.attention.self.key_proj.bias
Trainable layer: encoder.layer.0.attention.self.value_proj.weight
Trainable layer: encoder.layer.0.attention.self.value_proj.bias
Trainable layer: encoder.layer.0.attention.output.dense.weight
Trainable layer: encoder.layer.0.attention.output.dense.bias
Trainable layer: encoder.layer.0.attention.output.LayerNorm.weight
Trainable layer: encoder.layer.0.attention.output.LayerNorm.bias
Trainable layer: encoder.layer.0.intermediate.dense.weight
Trainable layer: encoder.layer.0.intermediate.dense.bias
Trainable layer: encoder.layer.0.output.dense.weight
Trainable layer: encoder.layer.0.output.dense.bias

In [10]:
text = "I worked with React.js and Docker on cloud deployment"

inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True
)

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


torch.Size([1, 16, 768])


In [11]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

['[CLS]', '▁I', '▁work', 'ed', '▁with', '▁React', '.', 'js', '▁and', '▁Dock', 'er', '▁on', '▁cloud', '▁de', 'ployment', '[SEP]']


In [12]:
!pip install pytorch-crf


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: D:\Deep_learning\portfolio_recommendation\venv\Scripts\python.exe -m pip install --upgrade pip


In [13]:
import gc
torch.cuda.empty_cache()
gc.collect()

107

In [14]:
import torch
import torch.nn as nn
from transformers import AutoModel

class MultitaskXLM(nn.Module):
    def __init__(
        self,
        model_name="xlm-roberta-base",
        num_domain_labels=10,
        num_ner_labels=3,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.domain_dropout = nn.Dropout(0.2)
        self.ner_dropout = nn.Dropout(0.1)
        self.domain_classifier = nn.Linear(hidden_size, num_domain_labels)
        self.ner_classifier = nn.Linear(hidden_size, num_ner_labels)

    def forward(self, input_ids, attention_mask, domain_labels=None, ner_labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        hidden_states = outputs.last_hidden_state.float()

        # cls_vector = hidden_states[:, 0, :]
        mask_expanded = attention_mask.unsqueeze(-1).float()
        mean_pooled = (hidden_states * mask_expanded).sum(1) / mask_expanded.sum(1)

        domain_logits = self.domain_classifier(self.domain_dropout(mean_pooled)) # instead of cls for better representation of the sequence
        ner_logits = self.ner_classifier(self.ner_dropout(hidden_states))

        loss = None

        if domain_labels is not None and ner_labels is not None:
            domain_loss_fn = nn.BCEWithLogitsLoss()
            ner_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

            domain_loss = domain_loss_fn(domain_logits, domain_labels.float())

            ner_loss = ner_loss_fn(
                ner_logits.reshape(-1, ner_logits.shape[-1]),
                ner_labels.reshape(-1)
            )

            loss = domain_loss * 0.7 + ner_loss * 0.3 # Weighted loss because NER has way higher results that Domain classification

        return {
            "domain_logits": domain_logits,
            "ner_logits": ner_logits,
            "loss": loss
        }

-------- TEST FOR PRODUCTION ---------


In [15]:
model = MultitaskXLM(num_domain_labels=10,model_name="microsoft/mdeberta-v3-base" , num_ner_labels=3)
model.to(device)

batch = next(iter(train_dataloader))

batch = {
    key: value.to(device) if torch.is_tensor(value) else value
    for key, value in batch.items()
}

outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    domain_labels=batch["domain_labels"],
    ner_labels=batch["ner_labels"],
)

print(outputs["loss"])
print(outputs["domain_logits"].shape)
print(outputs["ner_logits"].shape)

tensor(0.8228, device='cuda:0', grad_fn=<AddBackward0>)
torch.Size([8, 10])
torch.Size([8, 84, 3])


# Training helpers

In [17]:
import wandb
wandb.login()

AttributeError: module 'wandb.proto.wandb_internal_pb2' has no attribute 'Result'

In [ ]:
import torch
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr= 2e-5)
# No loss here because in the models class for each classifier head we define a different loss

-------------- FOR PRODUCTION --------------

In [ ]:
scaler = GradScaler('cuda')

def train_step(model, dataloader, optimizer, device, scheduler=None):
    model.train()
    train_loss = 0

    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        domain_labels = batch["domain_labels"].to(device)
        ner_labels = batch["ner_labels"].to(device)

        optimizer.zero_grad()

        with autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                domain_labels=domain_labels,
                ner_labels=ner_labels
            )
            loss = outputs["loss"]

        scaler.scale(loss).backward()
        # Unscale BEFORE clipping so clip sees real gradient magnitudes
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f"Step {batch_idx} | Loss {loss.item():.4f}")

    return train_loss / len(dataloader)

In [ ]:
from sklearn.metrics import f1_score
import torch

def eval_step(model, dataloader, device):
    model.eval()

    total_loss = 0

    all_domain_preds = []
    all_domain_labels = []

    all_ner_preds = []
    all_ner_labels = []

    with torch.inference_mode():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            domain_labels = batch["domain_labels"].to(device)
            ner_labels = batch["ner_labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                domain_labels=domain_labels,
                ner_labels=ner_labels,
            )

            loss = outputs["loss"]
            total_loss += loss.item()

            # Domain predictions
            domain_logits = outputs["domain_logits"]
            domain_preds = (torch.sigmoid(domain_logits) > 0.5).int()

            all_domain_preds.append(domain_preds.cpu())
            all_domain_labels.append(domain_labels.cpu().int())

            # NER predictions
            ner_logits = outputs["ner_logits"]
            ner_preds = ner_logits.argmax(dim=-1)

            mask = ner_labels != -100

            all_ner_preds.append(ner_preds[mask].cpu())
            all_ner_labels.append(ner_labels[mask].cpu())

    avg_loss = total_loss / len(dataloader)

    all_domain_preds = torch.cat(all_domain_preds).numpy()
    all_domain_labels = torch.cat(all_domain_labels).numpy()

    all_ner_preds = torch.cat(all_ner_preds).numpy()
    all_ner_labels = torch.cat(all_ner_labels).numpy()

    domain_micro_f1 = f1_score(
        all_domain_labels,
        all_domain_preds,
        average="micro",
        zero_division=0
    )

    domain_macro_f1 = f1_score(
        all_domain_labels,
        all_domain_preds,
        average="macro",
        zero_division=0
    )

    ner_micro_f1 = f1_score(
        all_ner_labels,
        all_ner_preds,
        average="micro",
        zero_division=0
    )

    ner_macro_f1 = f1_score(
        all_ner_labels,
        all_ner_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "domain_micro_f1": domain_micro_f1,
        "domain_macro_f1": domain_macro_f1,
        "ner_micro_f1": ner_micro_f1,
        "ner_macro_f1": ner_macro_f1,
    }

To evaluate our model we will use f1 score metrics, And to explain f1 score metrics we have to explain recall and precisions metrics first :

- Precision : Of all the items the model labeled as positive, how many were actually positive?
- Recall : Of all the actual positives, how many did the model correctly identify?
- F1 score : The harmonic mean of precision and recall. It balances the two metrics into a single number, making it especially useful when precision and recall are in trade-off.



> From Medium - Understanding Precision, Recall, and F1 Score Metrics

And in our specific use case, both uncorrect positive predictions and missed positive labels are important. With that said F1 Score is the best choice of evaluation.


In [ ]:
def compute_ner_token_f1(ner_logits, ner_labels):

  preds = ner_logits.argmax(dim=-1)

  mask = ner_logits != -100

  preds = preds[mask].cpu().numpy()
  labels = ner_labels[mask].cpu().numpy()

  micro_f1 = f1_score(labels ,preds ,average="micro" ,zero_division=0)
  macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)

  return micro_f1,macro_f1

In [ ]:
import wandb

wandb.init(
    project="portfolio-nlp",
    name="changing-encoder-prod-preprocessagain",
    config={
        "epochs": 10,
        "optimizer": "AdamW",
        "model": "microsoft/mdeberta-v3-base",
        "lr": "5e-5",
    }
)

In [ ]:

from tqdm.auto import tqdm
import torch

def train(
    model,
    train_dataloader,
    val_dataloader,
    optimizer,
    device,
    epochs=5,
    scheduler=None
):
    model.to(device)

    history = {
        "train_loss": [],
        "val_loss": [],
        "domain_f1": [],
        "ner_f1": []
    }

    best_val_loss = float("inf")

    for epoch in range(epochs):

        print(f"\n===== Epoch {epoch+1}/{epochs} =====")

        # Train
        train_loss = train_step(
            model=model,
            dataloader=train_dataloader,
            optimizer=optimizer,
            device=device,
            scheduler= scheduler,
        )

        # Validation
        val_metrics = eval_step(
            model=model,
            dataloader=val_dataloader,
            device=device
        )

        # for cosine annealing we do per epoch step
        if scheduler:
          scheduler.step()

        val_loss = val_metrics["loss"]

        # Save history
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["domain_f1"].append(val_metrics["domain_micro_f1"])
        history["ner_f1"].append(val_metrics["ner_micro_f1"])

        print(f"Train loss : {train_loss:.4f}")
        print(f"Val loss   : {val_loss:.4f}")
        print(f"Domain F1  : {val_metrics['domain_micro_f1']:.4f}")
        print(f"NER F1     : {val_metrics['ner_micro_f1']:.4f}")
        print(f"LR         : {optimizer.param_groups[0]['lr']}")

        #wandb logs
        lr = optimizer.param_groups[0]['lr']
        wandb.log({
            "epoch": epoch + 1,

            # losses
            "train/loss": train_loss,
            "val/loss": val_loss,

            # metrics
            "val/domain_f1": val_metrics["domain_micro_f1"],
            "val/ner_f1": val_metrics["ner_micro_f1"],

            # learning rate
            "lr": lr,
        })

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_model.pt")
            print("Best model saved.")

    return history

In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

optimizer = torch.optim.AdamW(model.parameters(), lr=4e-5)

EPOCHS = 10

total_steps = len(train_dataloader) * EPOCHS
warmup_steps = int(total_steps * 0.1)  # 10% warmup is standard

#scheduler = get_linear_schedule_with_warmup(
#    optimizer,
#    num_warmup_steps=warmup_steps,
#    num_training_steps=total_steps
#)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,        # one full cosine cycle over all epochs
    eta_min=1e-8         # minimum LR at the bottom of the curve
)

losses = train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader= validation_dataloader,
    optimizer=optimizer,
    device=device,
    epochs= EPOCHS,
    scheduler= scheduler,
)


===== Epoch 1/10 =====
Step 0 | Loss 1.0388
Step 10 | Loss nan
Step 20 | Loss nan
Step 30 | Loss nan


KeyboardInterrupt: 

In [ ]:
    wandb.finish()

In [ ]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("microsoft/mdeberta-v3-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

tex2 = "The Smart Retail Management System (SRMS) is a cloud-based enterprise application developed for medium-sized retail businesses to automate inventory management, sales tracking, customer relationship management, and business analytics."
text3 = "Developed a multilingual AI system based on Transformer architectures to automatically analyze and classify technical portfolio content. The project combines Named Entity Recognition (NER) and multi-label text classification in a single Multi-Task Learning (MTL) model using XLM-RoBERTa."
text4 = "Developed a scalable e-commerce platform using React, Next.js, and TypeScript for the frontend with a Node.js and Express backend. Integrated Redis caching and PostgreSQL for performance optimization. Containerized the application with Docker and deployed it on Kubernetes using Helm charts and GitHub Actions CI/CD pipelines. Implemented OAuth2 authentication, JWT session management, and Web Application Firewall protection with Cloudflare."
text5 = "Designed a smart irrigation system powered by ESP32 microcontrollers and MQTT communication. Sensor data for soil humidity and temperature was transmitted to AWS IoT Core and visualized through a Flutter mobile application. Implemented Bluetooth Low Energy pairing and offline synchronization for remote agricultural environments with unstable connectivity."
text= "Built a real-time anomaly detection pipeline using Python, Kafka, and Spark Structured Streaming, deployed on Kubernetes with Helm charts, monitored via Prometheus and Grafana, models trained with scikit-learn and tracked in MLflow, infrastructure provisioned by Terraform on AWS, with a React dashboard consuming a FastAPI backend secured by OAuth2."

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.inference_mode():
    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

domain_probs = torch.sigmoid(outputs["domain_logits"])[0]
domain_preds = (domain_probs > 0.1).int()

print(f"TEXT : {text} \n")
print(domain_probs)
print(domain_preds)

domain_names = [
    "Web Frontend",
    "Web Backend",
    "Mobile Development",
    "DevOps and Cloud Infrastructure",
    "Data Engineering",
    "Machine Learning and AI",
    "Cybersecurity",
    "Embedded Systems and IoT",
    "High Performance and Quantum Computing",
    "Other",
]

for i, pred in enumerate(domain_preds):
    if pred == 1:
        print(f"\n Domains predicted : {domain_names[i], float(domain_probs[i])}")

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

ner_preds = outputs["ner_logits"].argmax(dim=-1)[0].cpu().tolist()

id2label = {
    0: "O",
    1: "B-TECH",
    2: "I-TECH"
}

print (f"\n Technologies :")
for token, pred in zip(tokens, ner_preds):
    if pred != 0:
        print(token, id2label[pred])

TEXT : Built a real-time anomaly detection pipeline using Python, Kafka, and Spark Structured Streaming, deployed on Kubernetes with Helm charts, monitored via Prometheus and Grafana, models trained with scikit-learn and tracked in MLflow, infrastructure provisioned by Terraform on AWS, with a React dashboard consuming a FastAPI backend secured by OAuth2. 

tensor([0.0065, 0.1416, 0.0108, 0.9738, 0.8389, 0.9554, 0.0808, 0.1017, 0.1428,
        0.0038], device='cuda:0')
tensor([0, 1, 0, 1, 1, 1, 0, 1, 1, 0], device='cuda:0', dtype=torch.int32)

 Domains predicted : ('Web Backend', 0.1415930688381195)

 Domains predicted : ('DevOps and Cloud Infrastructure', 0.9738414883613586)

 Domains predicted : ('Data Engineering', 0.8388534784317017)

 Domains predicted : ('Machine Learning and AI', 0.9554203748703003)

 Domains predicted : ('Embedded Systems and IoT', 0.1016964241862297)

 Domains predicted : ('High Performance and Quantum Computing', 0.1427958756685257)

 Technologies :
Built B-T

In [ ]:
# load best model
model = MultitaskXLM(
    model_name="xlm-roberta-base",
    num_domain_labels=10,
    num_ner_labels=3,
)
model.load_state_dict(torch.load("best_model.pt"))
model.to(device)
model.eval()

# run on val or test dataloader
all_domain_preds = []
all_domain_labels = []

with torch.no_grad():
    for batch in validation_dataloader:  # or test_dataloader
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        domain_labels = batch["domain_labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        domain_preds = (torch.sigmoid(outputs["domain_logits"]) > 0.5).int()

        all_domain_preds.append(domain_preds.cpu())
        all_domain_labels.append(domain_labels.cpu().int())

all_domain_preds = torch.cat(all_domain_preds).numpy()
all_domain_labels = torch.cat(all_domain_labels).numpy()

from sklearn.metrics import classification_report
print(classification_report(
    all_domain_labels,
    all_domain_preds,
    target_names = [
    "Web Frontend",
    "Web Backend",
    "Mobile Development",
    "DevOps and Cloud Infrastructure",
    "Data Engineering",
    "Machine Learning and AI",
    "Cybersecurity",
    "Embedded Systems and IoT",
    "High Performance and Quantum Computing",
    "Other",
],
    zero_division=0
))

In [ ]:
from google.colab import files
files.download("best_model.pt")

# **Optimizing From checkpoint**

In [ ]:
for name, module in model.named_children():
    print(name)

In [ ]:
import wandb

wandb.init(
    project="portfolio-nlp",
    name="FINETUNING-PHASE2-mixedLR",
    config={
        "epochs": 8,
        "optimizer": "AdamW",
        "model": "xlm-roberta-base",
        "lr": "mixed",
    }
)

In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

optimizer = torch.optim.AdamW([
    {
        "params": model.encoder.parameters(),
        "lr": 1e-6,          # encoder is already well adapted
        "weight_decay": 0.01
    },
    {
        "params": model.domain_classifier.parameters(),
        "lr": 2e-5,          # weakest part of the model
        "weight_decay": 0.01
    },
    {
        "params": model.ner_classifier.parameters(),
        "lr": 8e-6,          # already strong
        "weight_decay": 0.01
    },
])

EPOCHS = 8  # shorter phase

total_steps = len(train_dataloader) * EPOCHS
warmup_steps = int(total_steps * 0.1)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-8
)

losses = train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader= validation_dataloader,
    optimizer=optimizer,
    device=device,
    epochs= EPOCHS,
    scheduler= scheduler,
)

In [ ]:
wandb.finish()